# F3-C — Baseline: LogisticRegression sobre embeddings DistilBERT

**Objetivo**: Cargar embeddings frozen de F3-A y entrenar una regresión logística como baseline. Incluye visualizaciones (t-SNE, UMAP, K-Means, similitud coseno).

**Tiempo estimado**: ~5 min (CPU)


## 1. Instalar dependencias


In [ ]:
!pip install -q mlflow umap-learn


In [ ]:
import numpy as np
import os
import json
import time
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             accuracy_score, confusion_matrix,
                             classification_report)
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics.pairwise import cosine_similarity
import umap
import joblib
import gc


## 2. Montar Google Drive y cargar embeddings


In [ ]:
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/ML/proyecto_integrador"
EMB_DIR = f"{DRIVE_BASE}/embeddings"
REPORTS_DIR = f"{DRIVE_BASE}/reports"
RANDOM_STATE = 42

for d in [REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Cargando embeddings desde F3-A...")
X_train_emb = np.load(f"{EMB_DIR}/train_embeddings.npy")
X_val_emb   = np.load(f"{EMB_DIR}/val_embeddings.npy")
X_test_emb  = np.load(f"{EMB_DIR}/test_embeddings.npy")
y_train = np.load(f"{EMB_DIR}/train_labels.npy")
y_val   = np.load(f"{EMB_DIR}/val_labels.npy")
y_test  = np.load(f"{EMB_DIR}/test_labels.npy")

print(f"Train embeddings: {X_train_emb.shape}")
print(f"Val embeddings:   {X_val_emb.shape}")
print(f"Test embeddings:  {X_test_emb.shape}")


## 3. Logistic Regression (baseline)

Regresión logística sobre embeddings de 768d. Baseline rápido.


In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)

start = time.time()
clf.fit(X_train_emb, y_train)
training_time = time.time() - start
print(f"Entrenado en {training_time:.1f}s")

y_pred = clf.predict(X_test_emb)

report = classification_report(y_test, y_pred,
                               target_names=['Negativo', 'Neutro', 'Positivo'],
                               output_dict=True)
per_class = {cl: report[cl] for cl in ['Negativo', 'Neutro', 'Positivo']}

metrics = {
    'model_name': 'DistilBERT + LogisticRegression',
    'model_type': 'distilbert_base_uncased + logreg',
    'sample_size': 200_000,
    'embedding_dim': 768,
    'training_time_seconds': round(training_time, 2),
    'f1_macro': round(report['macro avg']['f1-score'], 4),
    'precision_macro': round(report['macro avg']['precision'], 4),
    'recall_macro': round(report['macro avg']['recall'], 4),
    'accuracy': round(accuracy_score(y_test, y_pred), 4),
    'class_labels': ['Negativo', 'Neutro', 'Positivo'],
    'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
    'per_class': per_class,
}

for k, v in metrics.items():
    if k not in ('confusion_matrix', 'per_class'):
        print(f"  {k}: {v}")


## 4. MLflow Tracking


In [ ]:
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "https://humorous-trusting-domelike.ngrok-free.dev")
import requests
try:
    r = requests.get(f"{MLFLOW_TRACKING_URI}/api/2.0/mlflow/experiments/list", timeout=5)
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    print(f"MLflow OK via {MLFLOW_TRACKING_URI}")
except Exception as e:
    print(f"MLflow no disponible: {e}, fallback a SQLite")
    mlflow.set_tracking_uri(f"sqlite:///{DRIVE_BASE}/mlflow_fallback.db")

mlflow.set_experiment("distilbert_baseline")

with mlflow.start_run():
    mlflow.log_params({
        "model_name": "DistilBERT + LogisticRegression",
        "sample_size": 200_000,
        "embedding_dim": 768,
    })
    for label, scores in metrics['per_class'].items():
        mlflow.log_metric(f'f1_{label.lower()}', scores['f1-score'])
    mlflow.log_metrics({
        'f1_macro': metrics['f1_macro'],
        'accuracy': metrics['accuracy'],
        'training_time_seconds': metrics['training_time_seconds'],
    })
    MODEL_PATH = f"{EMB_DIR}/classifier.pkl"
    joblib.dump(clf, MODEL_PATH)
    mlflow.log_artifact(MODEL_PATH, artifact_path="models")
    print(f"MLflow run ID: {mlflow.active_run().info.run_id}")


## 5. Exportar métricas a JSON


In [ ]:
report_path = f"{REPORTS_DIR}/metrics_distilbert.json"
with open(report_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Exportado: {report_path}")


## 6. Guardar embeddings para F4 (RAG)


In [ ]:
EMBEDDINGS_PATH = f"{EMB_DIR}/distilbert_embeddings_sample.npy"
LABELS_PATH = f"{EMB_DIR}/distilbert_labels_sample.npy"

# Si F3-A ya los guardó, verificar
if os.path.exists(EMBEDDINGS_PATH):
    print("Embeddings para F4 ya existen (de F3-A), omitiendo...")
else:
    emb_sample_size = 10_000
    n_per_class = emb_sample_size // 3
    rng = np.random.RandomState(RANDOM_STATE)
    emb_chunks, label_chunks = [], []
    for s in [0, 1, 2]:
        idx = np.where(y_test == s)[0]
        n = min(n_per_class, len(idx))
        chosen = rng.choice(idx, n, replace=False)
        emb_chunks.append(X_test_emb[chosen])
        label_chunks.append(y_test[chosen])
    emb_sample = np.concatenate(emb_chunks)
    label_sample = np.concatenate(label_chunks)
    np.save(EMBEDDINGS_PATH, emb_sample)
    np.save(LABELS_PATH, label_sample)
    print(f"Saved {len(emb_sample)} embeddings para F4")


## 7. Visualizaciones

Matriz de confusion, t-SNE, UMAP, K-Means clustering, similitud coseno.


### 7a. Matriz de confusion


In [ ]:
cm = np.array(metrics['confusion_matrix'])
labels_names = metrics['class_labels']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_names, yticklabels=labels_names)
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title('Matriz de Confusión - DistilBERT')
plt.tight_layout()
plt.show()


### 7b. t-SNE (5k muestras)


In [ ]:
tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30)
emb_tsne = tsne.fit_transform(X_test_emb[:5000])

plt.figure(figsize=(10, 8))
palette = ['#e74c3c', '#f39c12', '#2ecc71']
for label in [0, 1, 2]:
    mask = y_test[:5000] == label
    plt.scatter(emb_tsne[mask, 0], emb_tsne[mask, 1],
                c=palette[label], label=labels_names[label], alpha=0.6, s=10)
plt.title('t-SNE de Embeddings DistilBERT (5k muestras)')
plt.legend()
plt.tight_layout()
plt.show()


### 7c. UMAP


In [ ]:
reducer = umap.UMAP(n_neighbors=50, min_dist=0.1, n_components=2,
                   random_state=RANDOM_STATE, init='random')
emb_umap = reducer.fit_transform(X_test_emb)

plt.figure(figsize=(8, 6))
for s in [0, 1, 2]:
    mask = y_test == s
    plt.scatter(emb_umap[mask, 0], emb_umap[mask, 1],
                c=palette[s], label=labels_names[s], alpha=0.5, s=8)
plt.title('UMAP: Embeddings DistilBERT')
plt.legend()
plt.tight_layout()
plt.show()


### 7d. K-Means clustering


In [ ]:
K_range = range(2, 9)
inertias, sil_scores, db_scores, ch_scores = [], [], [], []

rng = np.random.RandomState(RANDOM_STATE)
idx_sub = rng.choice(len(X_test_emb), 5_000, replace=False)
emb_sub = X_test_emb[idx_sub]

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(emb_sub)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(emb_sub, labels))
    db_scores.append(davies_bouldin_score(emb_sub, labels))
    ch_scores.append(calinski_harabasz_score(emb_sub, labels))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0, 0].plot(list(K_range), inertias, marker='o')
axes[0, 0].set_title('Inercia (codo)')
axes[0, 1].plot(list(K_range), sil_scores, marker='o', color='green')
axes[0, 1].set_title('Silhouette Score (mayor = mejor)')
axes[1, 0].plot(list(K_range), db_scores, marker='o', color='red')
axes[1, 0].set_title('Davies-Bouldin (menor = mejor)')
axes[1, 1].plot(list(K_range), ch_scores, marker='o', color='purple')
axes[1, 1].set_title('Calinski-Harabasz (mayor = mejor)')
for ax in axes.flat:
    ax.set_xlabel('K')
plt.tight_layout()
plt.show()


### 7e. Similitud coseno


In [ ]:
n_per_class = 100
emb_list, txt_list, lbl_list = [], [], []
for s in [0, 1, 2]:
    idx_s = np.where(y_test == s)[0]
    chosen = np.random.RandomState(RANDOM_STATE).choice(idx_s, n_per_class, replace=False)
    emb_list.append(X_test_emb[chosen])
    lbl_list.append(y_test[chosen])

emb_cos = np.concatenate(emb_list)
sim_matrix = cosine_similarity(emb_cos)

plt.figure(figsize=(8, 6))
sns.heatmap(sim_matrix, cmap='coolwarm', vmin=0, vmax=1,
            xticklabels=False, yticklabels=False)
plt.title('Similitud Coseno entre Embeddings')
plt.tight_layout()
plt.show()


In [ ]:
# Liberar memoria
del X_train_emb, X_val_emb, X_test_emb, y_train, y_val, y_test
gc.collect()
print("\nF3-C completado.")
